1. Module install

In [0]:

# install world bank module

%pip install wbgapi

2. Fetch GDP data

In [0]:
import pandas as pd
import wbgapi as wb
from pyspark.sql import functions as F

# 1. Define country codes (USA, Euro Area 'EMU', China, Japan, India)
countries = ["USA", "EMU", "CHN", "JPN", "IND"]
indicator = "NY.GDP.MKTP.KD.ZG"

# 2. Dynamically find the maximum Year from your Silver table
max_year_row = spark.sql("SELECT MAX(CAST(Year AS INT)) AS max_year FROM db_miniproject_finance_ws.silver.gdp_macro_cleaned").collect()
latest_loaded_year = max_year_row[0]["max_year"]
target_year = latest_loaded_year + 1
print(f"Fetching next target year for incremental load: {target_year}")

# 3. Fetch data for the single next year from World Bank API
raw_gdp = wb.data.DataFrame(indicator, countries, time=target_year, skipBlanks=True)
gdp_df = raw_gdp.reset_index()

# 4. Convert Pandas DataFrame to Spark DataFrame
gdp_spark_df = spark.createDataFrame(gdp_df)

# 5. Normalize into the clean long format (economy, Year, GDP_Growth_Pct)
# wbgapi labels single-year columns with 'YR{year}'
data_col = [c for c in gdp_spark_df.columns if c != 'economy'][0]

incremental_long_df = (
    gdp_spark_df
    .withColumnRenamed(data_col, "GDP_Growth_Pct")
    .withColumn("Year", F.lit(target_year).cast("int"))
    .withColumn("GDP_Growth_Pct", F.col("GDP_Growth_Pct").cast("decimal(18,4)"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .select("economy", "Year", "GDP_Growth_Pct", "ingestion_timestamp")
)

# 6. Append the new row records directly to your Bronze table
(
    incremental_long_df.write.format("delta")
    .mode("append")
    .saveAsTable("bronze.gdp_macro_raw")
)

print(f"GDP macro data for {target_year} successfully appended in long format to bronze.gdp_macro_raw!")